In [1]:
import pandas as pd
import numpy as np

In [2]:
def select_best_ensemble_k(model_files, k=3, keep_top_n=7):
    """
    model_files: 字典 {模型名稱: 檔案路徑}
    k: 最終要選出來投票的模型數量
    keep_top_n: 初始篩選時要保留的前幾個模型 (依正確率排序)
    """
    
    results_wrong_vec = {} # 儲存每個模型是否答錯的向量 (1為錯, 0為對)
    accuracies = {}
    
    # 1. 讀取與計算基本數據
    for name, path in model_files.items():
        df = pd.read_excel(path)
        # 確保資料格式一致，避免 string vs int 的比對錯誤
        actual = df['Actual'].astype(str).str.strip()
        predicted = df['Predicted'].fillna(0).astype(int).astype(str).str.strip()
        
        # 計算正確率
        is_correct = (actual == predicted)
        accuracies[name] = is_correct.mean()
        
        # 記錄答錯位置 (True=1, False=0)
        results_wrong_vec[name] = (~is_correct).astype(int).values
    
    # 2. 依照正確率排序並篩選
    # 這裡我們用參數 keep_top_n 來決定保留多少模型
    sorted_by_acc = sorted(accuracies.items(), key=lambda x: x[1], reverse=True)
    top_models = [m[0] for m in sorted_by_acc[:keep_top_n]]
    
    print(f"--- 模型篩選結果 (保留前 {keep_top_n} 名) ---")
    for name in top_models:
        print(f"{name}: {accuracies[name]:.2%}")

    # 3. 計算雙重錯誤率矩陣
    num_questions = len(next(iter(results_wrong_vec.values())))
    de_matrix = pd.DataFrame(index=top_models, columns=top_models)

    for m1 in top_models:
        for m2 in top_models:
            if m1 == m2:
                de_matrix.loc[m1, m2] = np.nan
            else:
                # 兩者同時答錯 (向量相乘或用 &)
                both_wrong = np.sum((results_wrong_vec[m1] == 1) & (results_wrong_vec[m2] == 1))
                de_matrix.loc[m1, m2] = both_wrong / num_questions

    # 儲存矩陣
    de_matrix.to_csv("double_error_matrix.csv")

    # 4. 貪婪演算法選出 K 個模型 (加入偵錯輸出)
    selected = [top_models[0]] 
    remaining = [m for m in top_models if m not in selected]

    print(f"\n--- 迭代選擇過程 ---")
    print(f"Round 1: 初始基準模型 (正確率最高) = {selected[0]}")

    while len(selected) < k and remaining:
        scores = {}
        print(f"\nRound {len(selected) + 1} 評分中...")
        for candidate in remaining:
            # 詳列計算過程
            de_list = [de_matrix.loc[candidate, s] for s in selected]
            avg_de = np.mean(de_list)
            scores[candidate] = avg_de
            print(f"  > 候選 {candidate}: 與 {selected} 的錯誤率分別為 {de_list}, 平均 = {avg_de:.6f}")
        
        best_candidate = min(scores, key=scores.get)
        print(f"決策：選擇平均錯誤率最低的 [{best_candidate}]")
        
        selected.append(best_candidate)
        remaining.remove(best_candidate)
    return selected, de_matrix

In [3]:
# 這裡放入你的模型名稱與對應檔案
MODELS = ["gemma3_12b", "qwen3_8b", "deepseek-r1_8b", "llama3.1_8b", "mistral_7b", "huihui_ai_internlm3-abliterated", 
          "ingu627_phi-4-mini-instruct-q4", "glm4_9b", "myaniu_minicpm4-8b_Q4_K_M"]
# 建立範例檔案路徑 (請確保這些 xlsx 檔案存在)
MODEL_FILE_MAP = {m: f"result_{m}.xlsx" for m in MODELS}

# 執行選擇邏輯
final_models, full_matrix = select_best_ensemble_k(MODEL_FILE_MAP, k=3, keep_top_n=9)
print(f"\n最終建議投票的組合: {final_models}")

--- 模型篩選結果 (保留前 9 名) ---
qwen3_8b: 96.22%
glm4_9b: 95.73%
gemma3_12b: 95.66%
huihui_ai_internlm3-abliterated: 95.05%
deepseek-r1_8b: 94.51%
llama3.1_8b: 92.61%
myaniu_minicpm4-8b_Q4_K_M: 89.95%
mistral_7b: 87.85%
ingu627_phi-4-mini-instruct-q4: 85.48%

--- 迭代選擇過程 ---
Round 1: 初始基準模型 (正確率最高) = qwen3_8b

Round 2 評分中...
  > 候選 glm4_9b: 與 ['qwen3_8b'] 的錯誤率分別為 [np.float64(0.019835680751173708)], 平均 = 0.019836
  > 候選 gemma3_12b: 與 ['qwen3_8b'] 的錯誤率分別為 [np.float64(0.020539906103286387)], 平均 = 0.020540
  > 候選 huihui_ai_internlm3-abliterated: 與 ['qwen3_8b'] 的錯誤率分別為 [np.float64(0.019131455399061033)], 平均 = 0.019131
  > 候選 deepseek-r1_8b: 與 ['qwen3_8b'] 的錯誤率分別為 [np.float64(0.028051643192488263)], 平均 = 0.028052
  > 候選 llama3.1_8b: 與 ['qwen3_8b'] 的錯誤率分別為 [np.float64(0.02312206572769953)], 平均 = 0.023122
  > 候選 myaniu_minicpm4-8b_Q4_K_M: 與 ['qwen3_8b'] 的錯誤率分別為 [np.float64(0.023708920187793428)], 平均 = 0.023709
  > 候選 mistral_7b: 與 ['qwen3_8b'] 的錯誤率分別為 [np.float64(0.025117370892018778)], 平均 = 0.025117


In [4]:
full_matrix

,qwen3_8b,glm4_9b,gemma3_12b,huihui_ai_internlm3-abliterated,deepseek-r1_8b,llama3.1_8b,myaniu_minicpm4-8b_Q4_K_M,mistral_7b,ingu627_phi-4-mini-instruct-q4
qwen3_8b,NaN,0.019836,0.02054,0.019131,0.028052,0.023122,0.023709,0.025117,0.025704
glm4_9b,0.019836,NaN,0.021714,0.022066,0.022653,0.026526,0.026291,0.028286,0.029812
gemma3_12b,0.02054,0.021714,NaN,0.021596,0.023005,0.025352,0.025235,0.029225,0.028638
huihui_ai_internlm3-abliterated,0.019131,0.022066,0.021596,NaN,0.021479,0.028286,0.030164,0.030516,0.034507
deepseek-r1_8b,0.028052,0.022653,0.023005,0.021479,NaN,0.027817,0.028521,0.031925,0.03216
llama3.1_8b,0.023122,0.026526,0.025352,0.028286,0.027817,NaN,0.036033,0.045775,0.047183
myaniu_minicpm4-8b_Q4_K_M,0.023709,0.026291,0.025235,0.030164,0.028521,0.036033,NaN,0.044836,0.051291
mistral_7b,0.025117,0.028286,0.029225,0.030516,0.031925,0.045775,0.044836,NaN,0.063967
ingu627_phi-4-mini-instruct-q4,0.025704,0.029812,0.028638,0.034507,0.03216,0.047183,0.051291,0.063967,NaN
